# 13 — Merge All Job Sources

Merges all 11 per-source standardized CSVs into `final_jobs_dataset.csv`.

**Input:** `data/processed/jobs/*_standardized.csv` (11 files)
**Output:** `data/processed/jobs/final_jobs_dataset.csv`

> Only re-run if any individual source CSV has been updated.

In [ ]:
from pathlib import Path
import os, re
import pandas as pd

BASE_DIR = Path.cwd().parent.parent
JOBS_DIR = BASE_DIR / "data" / "processed" / "jobs"

## Schema and constants

In [ ]:
AGGREGATORS = {"linkedin", "staff.am", "job.am"}

OUT_COLS = [
    "source", "source_type", "source_url", "job_title", "company_name", "location",
    "employment_type", "seniority_level", "industries",
    "posting_date", "deadline", "skills_tags", "full_text",
]

## Helper functions

In [ ]:
def normalize_date(val):
    """Normalize various date formats to YYYY-MM-DD; blank on failure."""
    if not val or (isinstance(val, float)):
        return ""
    s = str(val).strip()
    # ISO with possible missing zero-padding: 2026-3-9 → 2026-03-09
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})", s)
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    # M/D/YYYY
    m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", s)
    if m:
        return f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    return s


def seniority_from_title(title: str) -> str:
    """Derive a seniority label from the job title when not provided."""
    t = str(title).lower()
    if any(k in t for k in ["intern", "trainee", "student"]):
        return "Internship"
    if any(k in t for k in ["principal", "staff ", "distinguished", "fellow"]):
        return "Principal"
    if any(k in t for k in ["senior", "sr.", "sr "]):
        return "Senior"
    if any(k in t for k in ["junior", "jr.", "jr "]):
        return "Junior"
    if any(k in t for k in ["lead", "head of", "manager", "director", "vp ", "architect"]):
        return "Lead"
    return ""


def read(filename):
    return pd.read_csv(JOBS_DIR / filename, dtype=str).fillna("")


def to_canonical(df: pd.DataFrame, **overrides) -> pd.DataFrame:
    """Return a DataFrame with exactly OUT_COLS, filling missing with ''."""
    n = len(df)
    data = {}
    for col in OUT_COLS:
        if col in overrides:
            val = overrides[col]
            # Series → use its values; scalar string → broadcast
            data[col] = val.values if isinstance(val, pd.Series) else [str(val)] * n
        elif col in df.columns:
            data[col] = df[col].values
        else:
            data[col] = [""] * n
    return pd.DataFrame(data)

## Load and standardize each source

In [ ]:
# ---------------------------------------------------------------------------
# 1. LinkedIn  (992 rows)
# ---------------------------------------------------------------------------

li_raw = read("linkedin_jobs_standardized.csv")
print(f"LinkedIn     : {len(li_raw):4d} rows")

li = to_canonical(
    li_raw,
    source          = "linkedin",
    source_url      = li_raw["link"],
    job_title       = li_raw["title"],
    company_name    = li_raw["companyName"],
    location        = li_raw["location"],
    employment_type = li_raw["employmentType"],
    seniority_level = li_raw["seniorityLevel"],
    industries      = li_raw["industries"],
    posting_date    = li_raw["postedAt"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
    full_text       = li_raw["descriptionText"],
)

In [ ]:
# ---------------------------------------------------------------------------
# 2. Staff.am  (55 rows)
# ---------------------------------------------------------------------------

sa_raw = read("staffam_jobs_standardized.csv")
print(f"Staff.am     : {len(sa_raw):4d} rows")

sa = to_canonical(
    sa_raw,
    seniority_level = sa_raw["specialist_level"],
    industries      = "",
    posting_date    = sa_raw["posting_date"].apply(normalize_date),
    deadline        = sa_raw["deadline"].apply(normalize_date),
)

In [ ]:
# ---------------------------------------------------------------------------
# 3. job.am  (20 rows)
# ---------------------------------------------------------------------------

ja_raw = read("jobam_jobs_standardized.csv")
print(f"job.am       : {len(ja_raw):4d} rows")

ja = to_canonical(
    ja_raw,
    seniority_level = ja_raw["experience_level"],
    industries      = "",
    posting_date    = "",
    deadline        = ja_raw["deadline"].apply(normalize_date),
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 4. Picsart  (2 rows)
# ---------------------------------------------------------------------------

pi_raw = read("picsart_jobs_standardized.csv")
print(f"Picsart      : {len(pi_raw):4d} rows")

pi = to_canonical(
    pi_raw,
    employment_type = "",
    seniority_level = pi_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = pi_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 5. Krisp  (7 rows)
# ---------------------------------------------------------------------------

kr_raw = read("krisp_jobs_standardized.csv")
print(f"Krisp        : {len(kr_raw):4d} rows")

kr = to_canonical(
    kr_raw,
    employment_type = kr_raw["work_type"],
    seniority_level = kr_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = kr_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 6. ServiceTitan  (4 rows)
# ---------------------------------------------------------------------------

st_raw = read("servicetitan_jobs_standardized.csv")
print(f"ServiceTitan : {len(st_raw):4d} rows")

st = to_canonical(
    st_raw,
    employment_type = "",
    seniority_level = st_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = st_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 7. EPAM  (108 rows)
# ---------------------------------------------------------------------------

ep_raw = read("epam_jobs_standardized.csv")
print(f"EPAM         : {len(ep_raw):4d} rows")

ep = to_canonical(
    ep_raw,
    employment_type = "",
    seniority_level = ep_raw["seniority_level"],
    industries      = ep_raw["department"],
    posting_date    = ep_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = ep_raw["skills_tags"],
)

In [ ]:
# ---------------------------------------------------------------------------
# 8. SoftConstruct  (152 rows)
# ---------------------------------------------------------------------------

sc_raw = read("softconstruct_jobs_standardized.csv")
print(f"SoftConstruct: {len(sc_raw):4d} rows")

sc = to_canonical(
    sc_raw,
    employment_type = "",
    seniority_level = sc_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = sc_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 9. DISQO  (1 row)
# ---------------------------------------------------------------------------

dq_raw = read("disqo_jobs_standardized.csv")
print(f"DISQO        : {len(dq_raw):4d} rows")

dq = to_canonical(
    dq_raw,
    employment_type = "",
    seniority_level = dq_raw["job_title"].apply(seniority_from_title),
    industries      = "Technology",
    posting_date    = dq_raw["posting_date"].apply(normalize_date),
    deadline        = "",
    skills_tags     = "",
)

In [ ]:
# ---------------------------------------------------------------------------
# 10. Synopsys  (2 rows)
# ---------------------------------------------------------------------------

sy_raw = read("synopsys_jobs_standardized.csv")
print(f"Synopsys     : {len(sy_raw):4d} rows")

sy = to_canonical(
    sy_raw,
    posting_date    = sy_raw["posting_date"].apply(normalize_date),
    deadline        = sy_raw["deadline"].apply(normalize_date),
)

In [ ]:
# ---------------------------------------------------------------------------
# 11. DataArt  (5 rows)
# ---------------------------------------------------------------------------

da_raw = read("dataart_jobs_standardized.csv")
print(f"DataArt      : {len(da_raw):4d} rows")

da = to_canonical(
    da_raw,
    posting_date    = da_raw["posting_date"].apply(normalize_date),
    deadline        = da_raw["deadline"].apply(normalize_date),
)

## Merge and save

In [ ]:
merged = pd.concat([li, sa, ja, pi, kr, st, ep, sc, dq, sy, da], ignore_index=True)

# Add source_type: aggregator (broad boards) vs company_portal (direct career pages)
merged["source_type"] = merged["source"].apply(
    lambda s: "aggregator" if s in AGGREGATORS else "company_portal"
)

merged = merged[OUT_COLS]

# Ensure no NaN leaks through
merged = merged.fillna("")

out_path = JOBS_DIR / "final_jobs_dataset.csv"
merged.to_csv(out_path, index=False, encoding="utf-8")

print(f"\nMerged saved : {out_path}")
print(f"Total rows   : {len(merged)}")

## Summary statistics

In [ ]:
print()

# Source breakdown
print("=" * 55)
print(f"{'SOURCE':<16} {'ROWS':>6}  {'FULL_TEXT':>10}  {'SENIORITY':>10}")
print("=" * 55)
for src, grp in merged.groupby("source"):
    ft_filled  = grp["full_text"].replace("", pd.NA).notna().sum()
    sen_filled = grp["seniority_level"].replace("", pd.NA).notna().sum()
    print(f"  {src:<14} {len(grp):>6}  {ft_filled:>5}/{len(grp):<4}  {sen_filled:>5}/{len(grp)}")
print("=" * 55)
print(f"  {'TOTAL':<14} {len(merged):>6}")
print()

# Field coverage
print("=== FIELD COVERAGE ===")
for col in OUT_COLS:
    filled = merged[col].replace("", pd.NA).notna().sum()
    pct    = filled / len(merged) * 100
    print(f"  {col:20s}: {filled:5d}/{len(merged)} ({pct:.1f}%)")
print()

# full_text stats
ft = merged["full_text"].replace("", pd.NA).dropna().str.len()
print(f"full_text (non-blank) — n={len(ft)}  Min={ft.min():.0f}  Median={ft.median():.0f}  Max={ft.max():.0f}")